# Data Processing

## Import packages
Dependency Specification: pandas==3.0.5, duckdb==1.5.6

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

from utils import *

## Load Data

Load data and select features with an SQL query through the duckdb.

In [14]:
year_selected = "2026" # temporarily hardcoded df year

In [15]:
con = duckdb.connect("lol_predictors.duckdb")

df = con.execute(f"SELECT * FROM df_{year_selected}").df()

con.close()

## Feature Exclusions
Features that have zero correlations with the target feature `result` or that correlate with each other should be dropped to avoid multicollinearity.

In [ ]:
# Compute correlation matrix and define the correlations of the target feature
corr_mtx = df.corr(numeric_only=True)
target_corr = corr_mtx['result'].abs()

# Find and drop features where correlation around 0.00
zero_corr_cols = target_corr[target_corr < 0.01].index.tolist()
df = df.drop(columns=zero_corr_cols, errors='ignore')

print("Dropped zero-correlation features:", zero_corr_cols)

Dropped zero-correlation features: ['ckpm']


After remove the features with zero correlations with the target feature `result`, we will have to manually remove the features that have high correlation with each other to apply domain knowledge on which of the two correlated features should retain.

In [17]:
# Adjust list based on the features considered
cols_to_drop = [
                'wardskilled'
                # All deaths stats are removed because they mirror their opps kills and assists
                'opp_deathsat10', 'opp_deathsat15', 'opp_deathsat20', 'opp_deathsat25'
                # `golddiffat15` and `xpdiffat15` surprisingly has a 0.85 corr value. 'golddiffat15' is removed because it has high collinearity with other features
                'golddiffat15',
                # Ingame stats from other timestamps does correlate past 0.80 with each other except for the 20th minute mark
                'golddiffat20', 'xpdiffat20',
                # `golddiffat25` and `xpdiffat25` has a 0.89 corr value. `xpdiffat25` is removed because gold has more impact in the late game
                'xpdiffat25'
                ]

df = df.drop(columns=cols_to_drop, errors='ignore')

Assists are not yet removed even though they have high collinearity with kills since they will be later used to calculate a support player's strength.

Also, while opponent stats also mirror with a team's stats, they will be retained to calculate "diff" features later.

## Data Cleaning

For **data imputation**, only the feature `split` has missing values based on df_2014's EDA. We can use the previous value for each NaN since there's only two values, which are `Summer` and `Spring`, and they are sorted chronologically.

In [ ]:
df['split'] = df['split'].ffill()

We will **trim outliers** because even the feature `wardskilled` that has the highest outlier percentage only has `<2%` of outliers.

In [ ]:
num_cols = df.select_dtypes(include=["number"]).columns.tolist()

print(num_cols)

for col in num_cols:
    df = df[~detect_outliers(df, col).to_numpy()]

This concludes data cleaning of df_2014, which is the Lol Esports data on 2014 based on Oracle's Elixir. From the EDA, there are no categories in categorical features that needs fixing. 

## Post-Processing Data Audit

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
display(df.describe())

In [ ]:
data_profiling(df)

In [ ]:
# Limit numerical columns for description and visualization
numcols_to_drop = ['result', 'firstbaron', 'firstdragon']

# Limit categorical columns for description and visualization
catcols_to_drop = ['gameid', 'teamname']

In [ ]:
count_outliers(df, numcols_to_drop)

*Note that the IQR bounds are dynamically recalculated on each call. These remaining outliers are relative to the newly trimmed distribution. Since the new percentage of outliers is < 1%, it is negligible and safe to retain.*

In [ ]:
visualize_num_distribution(df, numcols_to_drop, nrows=8, ncols=4, figsize=(15, 30))

In [ ]:
# Visualization of the Splits Distribution
plt.figure(figsize=(12, 6))
sns.countplot(data=df, x='split')
plt.title('DISTRIBUTION OF SPLITS', fontsize=12, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
visualize_outliers(df, numcols_to_drop, nrows=8, ncols=4, figsize=(15, 30))

In [ ]:
# Plot correlation heatmap
plt.figure(figsize=(40, 40))
sns.heatmap(corr_mtx, cmap='coolwarm', annot=True, fmt='.2f')
plt.title('CORRELATION HEATMAP', fontsize=24, fontweight="bold")
plt.show()

In [ ]:
df.to_csv("df_processed.csv", index=False)